In [40]:
import random

import duckdb
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim

In [41]:
class MLP(nn.Module):
    def __init__(self, embedding_dim: int, dropout_rate: float):
        super().__init__()
        self.linear1 = nn.Linear(
            in_features=embedding_dim,
            out_features=embedding_dim,
        )
        self.relu = nn.ReLU()
        self.dropout1 = nn.Dropout(p=dropout_rate)
        self.linear2 = nn.Linear(
            in_features=embedding_dim,
            out_features=embedding_dim,
        )
        self.dropout2 = nn.Dropout(p=dropout_rate)

    def forward(self, x: Tensor) -> Tensor:
        x = self.linear1(x)
        x = self.relu(x)
        x = self.dropout1(x)
        x = self.linear2(x)
        x = self.dropout2(x)
        return x


class SASRecBlock(nn.Module):
    def __init__(
        self,
        embedding_dim: int,
        heads_num: int,
        dropout_rate: float,
    ):
        super().__init__()
        self.attention = nn.MultiheadAttention(
            embed_dim=embedding_dim,
            num_heads=heads_num,
            dropout=dropout_rate,
            batch_first=True,
        )
        self.layer_norm1 = nn.LayerNorm(embedding_dim)
        self.mlp = MLP(embedding_dim=embedding_dim, dropout_rate=dropout_rate)
        self.layer_norm2 = nn.LayerNorm(embedding_dim)

    def forward(
        self,
        x: Tensor,
        attn_mask: Tensor,
        key_padding_mask: Tensor,
    ) -> Tensor:
        attn_out, _ = self.attention(
            query=x,
            key=x,
            value=x,
            attn_mask=attn_mask,
            key_padding_mask=key_padding_mask,
            need_weights=False
        )
        x = self.layer_norm1(x + attn_out)
        mlp_out = self.mlp(x)
        x = self.layer_norm2(x + mlp_out)
        return x


class SASRec(nn.Module):
    def __init__(
        self,
        items_num: int,
        sequence_len=64,
        embedding_dim=64,
        blocks_num=2,
        heads_num=1,
        dropout_rate=0.2,
    ):
        super().__init__()
        self.sequence_len = sequence_len
        self.item_embedddings = nn.Linear(
            in_features=items_num,
            out_features=embedding_dim,
            bias=False,
        )
        self.position_embedddings = nn.Embedding(
            num_embeddings=sequence_len,
            embedding_dim=embedding_dim,
        )
        self.embedding_dropout = nn.Dropout(p=dropout_rate)
        self.blocks = nn.ModuleList([
            SASRecBlock(
                embedding_dim=embedding_dim,
                heads_num=heads_num,
                dropout_rate=dropout_rate,
            )
            for _ in range(blocks_num)
        ])

    def forward(self, sequence_batch: Tensor) -> Tensor:
        batch_size, sequence_len, _ = sequence_batch.shape
        sequence_vectors = self.item_embedddings(sequence_batch.float())
        positions = torch.arange(end=sequence_len, device=sequence_batch.device)
        positions = positions.unsqueeze(dim=0).expand(batch_size, -1)
        position_vectors = self.position_embedddings(positions)
        output_batch = self.embedding_dropout(sequence_vectors + position_vectors)
        mask = nn.Transformer.generate_square_subsequent_mask(sequence_len)
        attn_mask = (mask == float("-inf")).to(sequence_batch.device)
        key_padding_mask = (sequence_batch.sum(dim=-1) == 0)
        for block in self.blocks:
            output_batch = block(
                x=output_batch,
                attn_mask=attn_mask,
                key_padding_mask=key_padding_mask,
            )
        return output_batch

    def predict(self, user_vectors: Tensor, item_indices: Tensor) -> Tensor:
        item_vectors = F.embedding(
            input=item_indices,
            weight=self.item_embedddings.weight.T,
        )
        scores = (user_vectors * item_vectors).sum(dim=-1)
        return scores


In [42]:
Sequence_pairs = list[tuple[list[list[int]], list[list[int]]]]

class UserSequenceDataset(Dataset):
    def __init__(
        self,
        user_histories: list[list[list[int]]],
        items_num: int,
        max_sequence_len=50,
        stride=1,
        negatives_num=4,
        pad_label=-1,
    ) -> None:
        super().__init__()
        self.items_num = items_num
        self.max_sequence_len = max_sequence_len
        self.negatives_num = negatives_num
        self.history_chunks: list[list[list[int]]] = []
        self.pad_label = pad_label
        self._chunk_histories(user_histories, max_sequence_len, stride)

    def _chunk_histories(
            self,
            user_histories: list[list[list[int]]],
            sequence_len: int,
            stride: int,
    ) -> None:
        for history in user_histories:
            sessions_num = len(history)
            if sessions_num < 2:
                continue
            for chunk_start in range(0, sessions_num - 1, stride):
                chunk_end = min(chunk_start + sequence_len + 1, sessions_num)
                self.history_chunks.append(history[chunk_start:chunk_end])

    def __len__(self) -> int:
        return len(self.history_chunks)

    def __getitem__(self, index: int) -> tuple[Tensor, Tensor, Tensor]:
        history_chunk = self.history_chunks[index]

        input_sequence = history_chunk[:-1]
        target_sequence = history_chunk[1:]
        
        input_encoded = torch.zeros(self.max_sequence_len, self.items_num)
        target_ids = torch.zeros(
            self.max_sequence_len, 
            self.negatives_num + 1, 
            dtype=torch.long,
        )

        sequence_len = len(input_sequence)
        sequence_start = self.max_sequence_len - sequence_len

        target_labels = torch.full(
            (self.max_sequence_len,),
            fill_value=self.pad_label,
            dtype=torch.long,
        )
        target_labels[sequence_start:] = 0
        
        for sequence_ind in range(sequence_len):
            position = sequence_start + sequence_ind

            input_encoded[
                position,
                input_sequence[sequence_ind],
            ] = 1.0

            all_positives = target_sequence[sequence_ind]
            positive_id = random.choice(all_positives)
            negative_ids = self._sample_negatives(all_positives)
            target_ids[position] = torch.tensor([positive_id] + negative_ids)
        
        return input_encoded, target_ids, target_labels

    def _sample_negatives(
            self,
            all_positives: list[list[int]],
    ) -> None:
        negatives = []
        while len(negatives) < self.negatives_num:
            negative_id = random.randint(1, self.items_num - 1)
            if negative_id not in all_positives:
                negatives.append(negative_id)

        return negatives

In [43]:
import os

with duckdb.connect(os.environ["DB_PATH"], read_only=True) as db_conn:
    ratings_df = db_conn.execute("""
        SELECT *
        FROM ratings
            INNER JOIN movies USING (movie_id)
            INNER JOIN users USING (user_id)
    """).df()

In [44]:
ratings_df.head()

,user_id,movie_id,rating,timestamp,title,genres,gender,age,occupation,zip_code
0,1,1193,5,978300760,One Flew Over the Cuckoo's Nest (1975),Drama,F,1,10,48067
1,1,661,3,978302109,James and the Giant Peach (1996),Animation|Children's|Musical,F,1,10,48067
2,1,914,3,978301968,My Fair Lady (1964),Musical|Romance,F,1,10,48067
3,1,3408,4,978300275,Erin Brockovich (2000),Drama,F,1,10,48067
4,1,2355,5,978824291,"Bug's Life, A (1998)",Animation|Children's|Comedy,F,1,10,48067


In [45]:
def leave_p_out_split(
        df: pd.DataFrame,
        min_rating=4.0,
        rating_col="rating",
        user_col="user_id",
        time_col="timestamp",
        p=1,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    filtered_df = df[df[rating_col] >= min_rating]
    filtered_df = filtered_df.sort_values(by=[user_col, time_col])
    test_df = filtered_df.groupby(user_col).tail(p)
    train_df = filtered_df.drop(index=test_df.index)

    return train_df, test_df

In [46]:
train_df, test_df = leave_p_out_split(df=ratings_df)
train_df, val_df = leave_p_out_split(df=train_df)

In [47]:
groupby_user = ratings_df.groupby("user_id")

submission_time_range = groupby_user["timestamp"].max() - groupby_user["timestamp"].min()
hours_range = submission_time_range / (60 * 60)

In [48]:
hours_range.describe()

count     6040.000000
mean      2279.938151
std       5325.829038
min          0.029444
25%          0.335417
50%          1.130139
75%        561.493056
max      24791.053333
Name: timestamp, dtype: float64

In [49]:
same_submission_sizes = train_df.groupby(["user_id", "timestamp"]).size()

same_submission_sizes.describe()

count    322329.000000
mean          1.747302
std           1.180620
min           1.000000
25%           1.000000
50%           1.000000
75%           2.000000
max          23.000000
dtype: float64

In [50]:
(same_submission_sizes > 2).mean()

0.18426514523980156

In [51]:
def prepare_user_histories(
        sorted_df: pd.DataFrame,
        user_col="user_id",
        movie_col="movie_id",
        time_col="timestamp",
):
    groups = sorted_df.groupby([user_col, time_col])
    sessions = groups[movie_col].apply(list).reset_index()
    user_sequences = sessions.groupby(
        by="user_id",
    )["movie_id"].apply(list)
    return user_sequences.tolist()

In [52]:
user_histories = prepare_user_histories(train_df)

In [53]:
user_histories[0][:5]

[[3186], [1270, 1721, 1022], [1836], [3408], [2804, 1207]]

In [54]:
train_df.head(8)

,user_id,movie_id,rating,timestamp,title,genres,gender,age,occupation,zip_code
31,1,3186,4,978300019,"Girl, Interrupted (1999)",Drama,F,1,10,48067
22,1,1270,5,978300055,Back to the Future (1985),Comedy|Sci-Fi,F,1,10,48067
27,1,1721,4,978300055,Titanic (1997),Drama|Romance,F,1,10,48067
37,1,1022,5,978300055,Cinderella (1950),Animation|Children's|Musical,F,1,10,48067
36,1,1836,5,978300172,"Last Days of Disco, The (1998)",Drama,F,1,10,48067
3,1,3408,4,978300275,Erin Brockovich (2000),Drama,F,1,10,48067
7,1,2804,5,978300719,"Christmas Story, A (1983)",Comedy|Drama,F,1,10,48067
47,1,1207,4,978300719,To Kill a Mockingbird (1962),Drama,F,1,10,48067


In [55]:
max_movie_id = ratings_df["movie_id"].max()

train_dataset = UserSequenceDataset(
    user_histories=user_histories,
    items_num=max_movie_id + 1,
)

In [56]:
sample = train_dataset[0]
sample[0].shape, sample[1].shape, sample[2].shape

(torch.Size([50, 3953]), torch.Size([50, 5]), torch.Size([50]))

In [57]:
def train_model(
    model: SASRec,
    dataset: UserSequenceDataset,
    num_epochs=10,
    batch_size=256,
    learn_rate=0.001,
    weight_decay=1e-8,
    device="cuda",
):
    print(f"Starting training on {device}")
    
    dataloader = DataLoader(
        dataset=dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=2,
        pin_memory=True,
    )

    optimizer = optim.AdamW(
        params=model.parameters(),
        lr=learn_rate,
        weight_decay=weight_decay,
    )

    criterion = nn.CrossEntropyLoss(
        ignore_index=dataset.pad_label,
    )
    
    model.to(device)
    model.train()

    batches_num = len(dataloader)
    
    for epoch in range(num_epochs):
        total_loss = 0.0
        total_samples = 0
        
        for batch_ind, (
                input_batch, # (batch_size, max_sequence_len, items_num)
                target_batch, # (batch_size, max_sequence_len, 1 + negatives_num)
                labels_batch # (batch_size, max_sequence_len, )
        ) in enumerate(dataloader):
            batch_size = input_batch.size(0)

            input_batch = input_batch.to(device)
            target_batch = target_batch.to(device)
            labels_batch = labels_batch.to(device) # ()

            user_vectors = model(sequence_batch=input_batch) # (batch_size, max_sequence_len, embedding_dim)

            target_scores = model.predict(
                user_vectors.unsqueeze(2),
                target_batch,
            ) # (batch_size, max_sequence_len, 1 + negatives_num)

            targets_num = target_scores.size(-1)

            loss = criterion(
                target_scores.view(-1, targets_num),
                labels_batch.view(-1),
            )

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            batch_average_loss = loss.item()
            print(f"Batch {batch_ind}/{batches_num}, batch loss {batch_average_loss:.4f}")
            total_loss += loss.item() * batch_size
            total_samples += batch_size
            
        avg_loss = total_loss / total_samples
        print(f"Epoch {epoch+1}/{num_epochs}, Average loss: {avg_loss:.4f}")

In [59]:
one_head_model = SASRec(items_num=max_movie_id + 1, heads_num=1)
train_model(model=one_head_model, dataset=train_dataset)

Starting training on cuda


Batch 0/1236, batch loss 1.6113
Batch 1/1236, batch loss 1.6066
Batch 2/1236, batch loss 1.5991
Batch 3/1236, batch loss 1.5892
Batch 4/1236, batch loss 1.5780
Batch 5/1236, batch loss 1.5607
Batch 6/1236, batch loss 1.5438
Batch 7/1236, batch loss 1.5226
Batch 8/1236, batch loss 1.4947
Batch 9/1236, batch loss 1.4667
Batch 10/1236, batch loss 1.4332
Batch 11/1236, batch loss 1.4077
Batch 12/1236, batch loss 1.3679
Batch 13/1236, batch loss 1.3337
Batch 14/1236, batch loss 1.2902
Batch 15/1236, batch loss 1.2792
Batch 16/1236, batch loss 1.2505
Batch 17/1236, batch loss 1.2253
Batch 18/1236, batch loss 1.1987
Batch 19/1236, batch loss 1.1734
Batch 20/1236, batch loss 1.1620
Batch 21/1236, batch loss 1.1352
Batch 22/1236, batch loss 1.1180
Batch 23/1236, batch loss 1.1019
Batch 24/1236, batch loss 1.0963
Batch 25/1236, batch loss 1.0771
Batch 26/1236, batch loss 1.0834
Batch 27/1236, batch loss 1.0467
Batch 28/1236, batch loss 1.0527
Batch 29/1236, batch loss 1.0542
Batch 30/1236, batch